### Sokoban-Solver
Ziel ist es, ein Sokoban-Level

<img src='/files/images/sokoban_level1.png'>

```python
class Level(NamedTuple):
    title: str
    ncol: int
    nrow: int
    player_pos: tuple(int, int)
    boxes: Iterable
    targets: Iterable
    blocked: Iterable
```
zu l&ouml;sen. Dazu durchsuchen wir mit BFS alle von der Startposition erreichbaren Spielzust&auml;nde.
Wir suchen einen Zustand, bei dem alle Boxen auf den Zielfeldern liegen.
Eine Greedy-Search Variante benutzt als Heuristik die Anzahl Box-Moves, um alle Boxen auf
ihre Zielfelder zu schieben.

Um den Suchraum klein zu halten, berechnen wir Folgendes:
- `box_cells`: Felder, von denen eine Box auf ein Zielfeld geschoben werden kann.  
  Um diese Felder zu finden, stellen wir eine Box auf ein Zielfeld und schauen, wohin wir diese Box **ziehen** k&ouml;nnen.
- `B`: Dict, der zu jedem Feld `pos` in `box_cells` angibt,
auf welche `box_cells` die Box geschoben werden kann (`B[pos]`).
- **Deadlocks**: Bilden 4 Boxen ein Quadrat, kann keine mehr je bewegt werden.  
  Das gleiche gilt, falls 1, 2 oder 3 Boxen zusammen mit blockierten Feldern ein Quadrat bilden.

  Ein Deadlock ist eine Menge `D` mit 1 bis 4 solcher Felder (die nicht nur aus Zielfeldern besteht).  
  Ist `D` eine Teilmenge der Menge `boxes`, sind die Boxen auf den Feldern `boxes & D` (Schnittmenge) f&uuml;r immer blockiert. 

    ```python
    def is_deadlock(boxes, deadlocks):
        for deadlock in deadlocks:
            if deadlock.issubset(boxes):
                return True
    ```  

**Der Suchraum**:  
Der Suchraum besteht aus allen Paaren (Boxpositionen, Spielerposition), wobei wir 
zwei solche Paare `(boxes_1, spieler_1)` und `(boxes_2, spieler_2)` als gleich ansehen, falls
- `boxes_1` und `boxes_2` Mengen mit den gleichen Elementen sind, und
- der Spieler von  Position `spieler_1` nach `spieler_2` ziehen kann, ohne Boxen verschieben zu müssen. 

Die exakte Spielerposition ist dabei irrelevant.
Deshalb repr&auml;sentieren wir den Suchraum als die Menge allen Paare
- (Boxpositionen, vom Spieler erreichbare Felder)

Bilden die `boxes` **keine** Barriere (der Spieler kann alle Felder ausser Boxfelder erreichen), dann
speichern wir statt der Felder nur `True`.

Wie Listen k&ouml;nnen auch Mengen nicht als Schlüssel in einem Dict verwendet werden.  
Wie ein Tuple kann aber ein `frozenset`, eine immutable (unveränderbare) Menge, ein Dict-Schlüssel sein.  
Beachte, dass im Gegensatz zu Tuple und Listen, folgender Vergleich `True` liefert.
```python
frozenset({1, 2}) == {2, 1}  # True
```

In [ ]:
# Level laden
import search_strategies as S
from level_loader import load_levels


levels = load_levels('Levels/Anomaly.txt')
level = levels[0]
level

Die **vom Spieler erreichbaren Felder** `cells` finden (Boxen ignorieren wir):  
Der Spieler kann sich in jede Richtung bewegen, ausser ein blockiertes Feld verhindert dies.  
Mit BFS suchen wir diese Felder.  

Damit die Nachbarn eines Feldes nicht jedesmal neu berechnet werden m&uuml;ssen, erstellen wir einen Dict  
`G`, der zu jedem erreichbaren Feld  `pos in cells`, die erreichbaren Felder `G[pos]` speichert.

In [ ]:
DIRECTIONS = ((1, 0), (-1, 0), (0, 1), (0, -1))


def get_neighbors(pos, blocked):
    '''liefert Nachbarfelder von pos, die nicht in blocked liegen'''
    for dc, dr in DIRECTIONS:
        if (new_pos := (pos[0]+dc, pos[1]+dr)) not in blocked:
            yield new_pos


def get_component(pos, get_neighbors):
    '''liefert die Menge der von pos aus erreichbaren Feldern.
       get_neighbors(pos) liefert die von pos in einem Schritt
       erreichbaren Felder.
    '''
    go_back = S.search_bf(pos, get_neighbors, None)[1]
    cells = set(go_back)
    return cells


get_ns = lambda pos: get_neighbors(pos, level.blocked)
cells = get_component(level.player_pos, get_ns)
len(cells)

In [ ]:
G = {pos: set(get_neighbors(pos, level.blocked)) for pos in cells}
G[level.player_pos]

In [ ]:
G

**Wohin k&ouml;nnen wir Boxen von Zielfelder ziehen (`box_cells`)**?  
Eine Box kann von (x, y) nach nach (x+1, y) gezogen werden, falls auch (x+2, y) vom Spieler erreichbar ist (ein Feld in `cells` ist).

Analog zu Dict `G` erstellen wir einen Dict `B`, der zu jedem Feld `pos` in `box_cells` angibt,
auf welche `box_cells` die Box geschoben werden kann. `B` erhalten wir aus `G` durch 
- entfernen aller Positionen ausserhalb `box_cells`,
- ein Box kann nur von Feld `p` auf `q` **geschoben** werden,
  falls sie von `q` noch `p` **gezogen** werden kann.  
  `q in B[p]` nur falls `p in get_pulls(q)`.

Eigentlich m&ouml;chten wir auch wissen, auf welchem Feld der Spieler sein muss,
um eine Box vom Feld `p` auf ein Feld `q in B[p]` schieben zu k&ouml;nnen.  
Der Dict `Bp` speichert zu jedem Feld `old_box_pos` eine Liste `Bp[old_box_pos]` von Tuplen `(player_pos, new_box_pos)`.





In [ ]:
def get_pulls(pos):
    for box in G[pos]:
        new_ppos = tuple(box[i] + (box[i] - pos[i])
                         for i in range(2))
        if new_ppos in cells:
            yield box


box_cells = get_component(level.targets[0], get_pulls)
len(box_cells)

In [ ]:
B1 = {pos: set(G[pos]) & box_cells for pos in box_cells}
B = {p: vs for p, ns in B1.items()
     if (vs := set(q for q in ns if p in get_pulls(q)))
     }
B = dict(sorted(B.items(), key=lambda x: (len(x[1]), x[0])))
B

In [ ]:
Bp = {}
for box, neighbors in B.items():
    Bp[box] = []
    for bpos in neighbors:
        ppos = tuple(box[i] + (box[i] - bpos[i])
                     for i in range(2))
        Bp[box].append((ppos, bpos))

Bp

**Deadlocks finden**:   
- Finde alle Deadlocks
- Sind $D_1\subset D_2$ Deadlocks, brauchen wir nur $D_1$.
- Da wir Boxen nur auf `box_cells` herumschieben, gen&uuml;gt es, Deadlocks zu betrachten,
  die Teilmengen von `box_cells` sind.

In [ ]:
def get_deadlocks(level, cells):
    deadlocks = []
    for c in range(level.ncol):
        for r in range(level.nrow):
            square = {(c, r), (c+1, r), (c, r+1), (c+1, r+1)}
            # deadlock-Quadrat hat Felder ausserhalb der targets
            if (deadlock := square & cells) and deadlock - set(level.targets):
                deadlocks.append(deadlock)

    return deadlocks


deadlocks = get_deadlocks(level, cells)
deadlocks[:3]

In [ ]:
def get_relevant_deadlocks(deadlocks, box_cells):
    deadlocks.sort(key=lambda x: len(x))

    n = len(deadlocks)
    to_keep = set(range(n))
    for i in range(n):
        for j in range(i+1, n):
            dl1, dl2 = deadlocks[i], deadlocks[j]
            if dl1.issubset(dl2):
                to_keep.discard(j)

    deadlocks = [deadlocks[i] for i in to_keep]
    return [deadlock for deadlock in deadlocks if deadlock.issubset(box_cells)]


relevant_deadlocks = get_relevant_deadlocks(deadlocks, box_cells)
relevant_deadlocks

**Nachbarzust&auml;nde finden**:

In [ ]:
def update_frozenset(fs, old_item, new_item):
    '''swap old_item for new_item'''
    s = set(fs)
    s.remove(old_item)
    s.add(new_item)
    return frozenset(s)


def get_comp_or_true(pos, blocked):
    comp = get_component(pos, lambda pos: G[pos] - blocked)
    return len(comp) == len(cells) - len(level.targets) or frozenset(comp)


def get_next_states(state, Bp):
    boxes, component = state  # component: True oder erreichbare Felder
    for box_pos in boxes:
        for player_pos, new_box_pos in Bp.get(box_pos, ()):
            if (
                new_box_pos not in boxes
                and player_pos not in boxes
                and (component is True or player_pos in component)
            ):
                new_boxes = update_frozenset(boxes, box_pos, new_box_pos)
                new_comp = get_comp_or_true(player_pos, new_boxes)
                yield new_boxes, new_comp


def get_box_move(state_1, state_2):
    '''liefert die Box Verschiebung zweier benachbarter Zustaende
       state_1=(boxes1, comp1) und state_2=(boxes2, comp2)
    '''
    old_box = list(state_1[0] - state_2[0])[0]
    new_box = list(state_2[0] - state_1[0])[0]
    return old_box, new_box

In [ ]:
initial_state = (frozenset(level.boxes), True)
next_states = list(get_next_states(initial_state, Bp))
for boxes, comp in next_states:
    if comp is not True:
        comp = set(comp)
    print(f'boxes: {set(boxes)}\ncomponent: {comp}')

In [ ]:
get_ns = lambda pos: get_neighbors(pos, level.blocked + level.targets)
comp = get_component((2, 2), get_ns)
goal_state = (frozenset(level.targets), frozenset(comp))
len(comp)

In [ ]:
from contextlib import redirect_stdout


with redirect_stdout(None):  # unterdruecke Ausgaben
    goal, go_back, _ = S.search_bf(initial_state, lambda s: get_next_states(s, Bp), goal_state)

In [ ]:
path = S.get_path_home(goal, go_back)[::-1]
n = len(path)

path[:2], n

In [ ]:
box_moves = [get_box_move(path[i], path[i+1]) for i in range(n-1)]
box_moves